In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Install the latest Google GenAI SDK
!pip -q install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 750.5 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 1.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 8.5 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.3/252.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 27.3 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.55.1 which is incompatible.
google-colab 1.0.

In [3]:
from kaggle_secrets import UserSecretsClient
from google import genai

# Read API Key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("GOOGLE_API_KEY")

# Create Gemini client
client = genai.Client(api_key=api_key)

# Test Gemini
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello! My AI Study Coach Agent is ready."
)

print(response.text)

Hello there! I'm ready too. Let's get started on your study goals!


In [5]:
# ==========================
# Memory
# ==========================

user_memory = {
    "name": "",
    "subject": "",
    "level": "",
    "goal": ""
}


def remember_user(name, subject, level, goal):
    user_memory["name"] = name
    user_memory["subject"] = subject
    user_memory["level"] = level
    user_memory["goal"] = goal


def show_memory():
    return user_memory

In [6]:
# ==========================
# Study Planner Tool
# ==========================

def create_study_plan():
    prompt = f"""
Create a personalized 4-week learning plan.

Student:
Name: {user_memory['name']}
Subject: {user_memory['subject']}
Level: {user_memory['level']}
Goal: {user_memory['goal']}

Return in markdown.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [7]:
# ==========================
# Quiz Tool
# ==========================

def create_quiz():
    prompt = f"""
Create 5 multiple-choice quiz questions.

Subject:
{user_memory['subject']}

Difficulty:
{user_memory['level']}

Include answers.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [8]:
# ==========================
# Adaptive Learning Agent
# ==========================

remember_user(
    name="Alex",
    subject="Python",
    level="Beginner",
    goal="Build AI Agents"
)

print("===== USER MEMORY =====")
print(show_memory())

print("\n===== STUDY PLAN =====")
print(create_study_plan())

print("\n===== QUIZ =====")
print(create_quiz())

===== USER MEMORY =====
{'name': 'Alex', 'subject': 'Python', 'level': 'Beginner', 'goal': 'Build AI Agents'}

===== STUDY PLAN =====
Here's a personalized 4-week learning plan for Alex, a beginner in Python, aiming to build AI Agents. This plan focuses on building a strong foundation in Python and then applying it to understand and create simple AI agents, primarily leveraging external APIs for AI capabilities due to the beginner level and short timeframe.

---

## Alex's 4-Week Python AI Agent Learning Plan

**Student:** Alex
**Subject:** Python
**Level:** Beginner
**Goal:** Build AI Agents

---

### **Overall Philosophy & Tips for Alex:**

1.  **Consistency is Key:** Dedicate at least 1-2 hours daily, or a few longer sessions (2-4 hours) several times a week.
2.  **Hands-on Practice:** Don't just read code; write it, break it, fix it. The projects are crucial.
3.  **Don't Get Stuck Alone:** Use online resources (Stack Overflow, Discord communities, official docs) when you encounter 

In [9]:
# ==========================
# Adaptive Feedback Tool
# ==========================

def adaptive_feedback(score, total=5):

    prompt = f"""
You are an AI Learning Coach.

Student Information:
Name: {user_memory['name']}
Subject: {user_memory['subject']}
Level: {user_memory['level']}
Goal: {user_memory['goal']}

The student scored {score}/{total} on the quiz.

Please provide:

1. Performance summary
2. Strengths
3. Weaknesses
4. What to study next
5. A revised one-week study plan

Format the response in Markdown.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [10]:
print("="*60)
print("ADAPTIVE FEEDBACK")
print("="*60)

# 假設學生答對 2 題
print(adaptive_feedback(score=2))

ADAPTIVE FEEDBACK
Hey Alex, great job taking the quiz! It's an excellent way to see where you're at and what we need to focus on next. Let's break down your performance and set you up for success.

---

### 1. Performance Summary

You scored 2 out of 5 on the recent quiz. This indicates that while you have some initial familiarity with the concepts, there are several fundamental areas of Python that need a bit more attention and practice. Don't worry, this is a very common starting point for beginners, and it simply tells us exactly where to concentrate your efforts to build a strong foundation for your goal of building AI Agents.

### 2. Strengths

*   **Motivation for AI Agents:** Your goal to build AI Agents is fantastic! This complex and exciting field requires a solid programming foundation, and having such a clear objective will keep you driven through the learning process.
*   **Engagement:** You attempted the quiz, which shows a willingness to assess your understanding and enga